# مقدمة لتحليل الشبكة باستخدام NetworkX
*بواسطة جورجي لازاريف* **(mlcourse slack: jorgy)**



الأنظمة منتشرة في كل مكان هذه الأيام في مختلف مجالات العلوم والصناعة ومواقف الحياة اليومية. يمكن تمثيل معظمها (على سبيل المثال لا الحصر، التفاعلات الاجتماعية وشبكات الهاتف والنقل) كشبكات أو رسوم بيانية (مجموعات من العقد والحواف حيث تميز العقد كائنات معينة وتشير الحواف إلى نوع من الاتصال بينها) وبالتالي فهي مناسبة للتحليل. 
قد نرغب في تحليل العلاقات بين المشاركين أو الجهات الفاعلة في تلك الأنظمة للحصول على بعض الأفكار القيمة:
- ما هي العقد الأكثر أهمية (المؤثرون في الشبكة)
- تحديد المسار - تحديد أقصر المسارات بين عقد معينة
- البنية - العثور على المجموعات، واكتشاف المجتمع (سنناقش هذه المصطلحات لاحقًا)
سأقدم لك في هذا البرنامج التعليمي حزمة NetworkX - Python من أجل "إنشاء ومعالجة ودراسة بنية وديناميكيات ووظائف الشبكات المعقدة". خصائص مهمة أشار إليها المطورون:
 - يوفر واجهة للخوارزميات الرقمية الموجودة والأكواد المكتوبة بلغات C وC++ وFORTRAN، مما يؤدي إلى أداء واعد للسرعة 
 - القدرة على العمل دون عناء مع مجموعات كبيرة من البيانات غير القياسية.
 
لنبدأ ببعض الأشياء الأساسية. في البداية تحتاج إلى تثبيت NetworkX (*'pip install Networkx'* أو *'conda install ..'* إذا كنت تعمل ضمن Anaconda)


In [ ]:
# importing necessary packages
import csv

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
import warnings
from collections import defaultdict

warnings.filterwarnings("ignore")

In [ ]:
# creating empty Graph
G = nx.Graph()

# you can now add nodes and edges
G.add_node("a")
G.add_node(1)
G.add_edge("a", 1)
# or creating list of nodes/edges in advance
G.add_nodes_from([2, 3, 4])
G.add_edges_from([(2, 4), ("a", 3)])

In [ ]:
# setting attribute while creating node
G.add_node("John", age=26)

# setting attribute to already existing node
G.node["John"]["hobby"] = "writing music"
nx.set_node_attributes(G, "number", 7)

# graph with random weghts
weights = {}
for e in G.edges():
    G[e[0]][e[1]]["weight"] = np.random.random()

In [ ]:
print(G.edges())
print(G.edges(data=True))
# data=True prints attributes too

In [ ]:
display(type(G))

In [ ]:
print(nx.info(G))

In [ ]:
nx.draw(G, with_labels=True, node_size=900)


من الجدير بالذكر أن استدعاء .draw دون تحديد الوسيطة *pos* سيؤدي إلى تخطيطات مختلفة للعقد.



## تحليل الرسم البياني الحقيقي


[هنا](http://konect.uni-koblenz.de/networks/) يمكنك العثور على شبكات حقيقية مخزنة كتمثيلات رسومية (قائمة الحواف في الغالب) بتنسيق .tsv. من أجل البساطة اخترت 
[رسم بياني] صغير جدًا وغير موجه (https://www.dropbox.com/s/4n86gethjw98gqk/out.dolphins?dl=0)، والذي يمثل تفاعلات اجتماعية متكررة بين 62 دلفينًا من مجتمع يعيش على صوت مشكوك فيه،
مضيق بحري في نيوزيلندا. لذلك هذا نوع من الشبكة الاجتماعية.
ليس لقراءة الرسوم البيانية من الملفات طريقة موحدة حيث يمكن تخزين البيانات بتنسيقات مختلفة ويمكن تمثيل الرسوم البيانية بشكل مختلف (مصفوفة الجوار، قوائم الحافة، قائمة المجاورة). أنصحك بالرجوع إليه [هنا](https://networkx.github.io/documentation/networkx-1.10/reference/convert.html) و[هنا](https://networkx.github.io/documentation/stable/reference/readwrite/index.html)


In [ ]:
edges = []
with open("datasets/out.dolphins", "r") as f:
    filereader = csv.reader(f, delimiter="\t", quotechar='"')
    next(filereader)  # we'll skip header row
    for row in filereader:
        edges.append(row[:2])
edges[:5]

In [ ]:
# now create graph from edgelist we made in previous step
GD = nx.from_edgelist(edges)

In [ ]:
print(nx.info(GD))

In [ ]:
nx.draw(GD, with_labels=True)


هناك أنواع أخرى من التصور باستثناء النوع المعتاد (مفيد بشكل خاص عند التعامل مع الرسوم البيانية الكبيرة) - ArcPlot وMatrixPlot وما إلى ذلك. سأعرض واحدًا فقط هنا - CircosPlot
لنقم بتثبيت واستيراد حزمة تصور رائعة لـ NetworkX - [nxviz](https://github.com/ericmjl/nxviz). 


In [ ]:
import nxviz as nv

In [ ]:
c = nv.CircosPlot(GD, node_labels=True)
c.draw()


كان ذلك سريعًا :-) الآن دعونا نتعمق في تحليل الرسم البياني الاجتماعي للدلافين


In [ ]:
# .has_edge method allows us to check presence of direct interaction between two nodes

GD.has_edge("40", "37")

In [ ]:
print(
    "shortest path from node 12 to 5 is of length",
    nx.shortest_path_length(GD, "12", "5"),
    ":",
    nx.shortest_path(GD, "12", "5"),
)


لكل قمة في الرسم البياني يمكننا حساب درجتها – عدد القمم المجاورة (المتصلة بحافة).


In [ ]:
display(list(GD.neighbors("9")))
print("node 9 has %d neighbors" % GD.degree()["9"])

In [ ]:
GD.degree()
# returns dictionary with all nodes

In [ ]:
# find 5 nodes with largest amount of neighbors
sorted(dict(GD.degree()).items(), key=lambda x: x[1], reverse=True)[:5]


وهذا يقودنا إلى عدة أنواع من المركزية - مقياس لوصف أهمية العقدة



### درجة المركزية



$$ degree   \  centrality = \frac {degree\ of\ node} {number\ of \ nodes\ in \ network} $$


In [ ]:
# We create dictionary with nodes and their degree centralities
degcent = nx.degree_centrality(GD)  # for getting dictionary with nodes and
# their degree centralities as keys and values respectively

print("Networkx degree centrality for node 16:", degcent["16"])


دعونا نرى 5 عقد ذات درجة مركزية أكبر:


In [ ]:
sorted(degcent.items(), key=lambda x: x[1], reverse=True)[:5]

In [ ]:
# we'll store node with the largest degree centraliy as 'nld '
nld = sorted(degcent.items(), key=lambda x: x[1], reverse=True)[:1][0][0]


هنا يأتي نوع من تصور هذه العقدة:


In [ ]:
palette = sns.color_palette("coolwarm", 2).as_hex()
palette

In [ ]:
# this list wi'll be used in .draw call.
node_colors = [palette[0] if n != nld else palette[1] for n in GD.nodes()]

In [ ]:
nx.draw(GD, node_color=node_colors, with_labels=True)


كما ذكر أعلاه، يمكنك تعيين سمات للعقد. لنفترض أننا نريد أن تحافظ كل عقدة على درجتها باعتبارها (واحدة من) السمة:


In [ ]:
for n in GD.nodes():
    GD.node[n]["degree centrality"] = degcent[n]


يمكننا الآن استخدامه في تخصيص تصورنا:


In [ ]:
c = nv.CircosPlot(GD, node_order="degree centrality", node_labels=True)
c.draw()


### مركزية القرب



$$closeness   \  centrality = \frac {N\ -\ 1} {sum\ of \ distances\ to \ each \ node \ from \ current \ one}$$
حيث $N = $ عدد العقد


يقارن عدد الخطوات (الحواف) التي قد يستغرقها الوصول إلى كل عقدة أخرى في الشبكة (باستخدام أقصر المسارات بشكل واضح).
يمكنك بشكل حدسي التفكير في هذا على أنه "متوسط ​​المسافة" لجميع العقد الأخرى.
تأخذ درجة المركزية في الاعتبار الروابط المباشرة فقط، مما يخدع المحلل في بعض الأحيان حيث أن العقدة الحالية يمكن أن تكون مركزية فقط في الحي المحلي. ولهذا السبب فإن مركزية التقارب مهمة.
في حالة عدم اتصال pf بشكل كامل، تقوم هذه الخوارزمية بحساب مركزية القرب لكل مكون متصل بشكل منفصل.
بالمناسبة، يمكنك رؤية الأجزاء المتصلة من الرسم البياني الخاص بك بهذه الطريقة (في حالتنا، الرسم البياني نفسه متصل):


In [ ]:
for comp in nx.connected_components(GD):
    print(comp)

In [ ]:
# calculating closeness centrality for each node
clocent = nx.closeness_centrality(GD)

In [ ]:
sorted(clocent.items(), key=lambda x: x[1], reverse=True)[:5]

In [ ]:
for n in GD.nodes():
    GD.node[n]["closeness centrality"] = clocent[n]

In [ ]:
nlcc = sorted(clocent.items(), key=lambda x: x[1], reverse=True)[:1][0][0]
palette = sns.color_palette("coolwarm", 2).as_hex()
node_colors = [palette[0] if n != nlcc else palette[1] for n in GD.nodes()]

In [ ]:
nx.draw(GD, node_color=node_colors, with_labels=True)


### بين المركزية



يمكننا حساب أقصر مسار لكل زوج من العقد (إذا كانت تنتمي إلى مكون واحد متصل بشكل واضح).
لذا تشير المركزية بين العقدة إلى عدد المرات التي تصبح فيها هذه العقدة الحالية جزءًا من جميع المسارات الأقصر.



$$betweeness   \  centrality = \frac {number \ of \ shortest \ paths \ passing \ throught \ current \ node } {totul \ number \ of \ shortest \ paths}$$



يمكن اعتبار المركزية بين الحدود كمقياس لتأثير العقدة في الشبكة. على سبيل المثال، يمكن لأعضاء المجموعة ذات التأثير "الكبير" أن يكونوا مفيدين من حيث نشر رسالة/معلومات. وكمثال آخر - في شبكة الاتصالات السلكية واللاسلكية، تتمتع العقدة ذات أكبر مركزية بينية بأكبر قدر من التحكم بسبب كمية المعلومات التي يتم نقلها عبرها.


In [ ]:
betcent = nx.betweenness_centrality(GD)
for n in GD.nodes():
    GD.node[n]["betweeness centrality"] = betcent[n]

sorted(betcent.items(), key=lambda x: x[1], reverse=True)[:5]

In [ ]:
nlbc = sorted(clocent.items(), key=lambda x: x[1], reverse=True)[:1][0][0]
palette = sns.color_palette("coolwarm", 2).as_hex()
node_colors = [palette[0] if n != nlbc else palette[1] for n in GD.nodes()]

In [ ]:
nx.draw(GD, node_color=node_colors, with_labels=True)


$Betweeness \ centrality \ for \ edges$
أعتقد أن هذا المصطلح لا يحتاج إلى شرح لأن معناه هو نفسه تقريبًا معنى العقدة


In [ ]:
betcent_e = nx.edge_betweenness_centrality(GD)
sorted(betcent_e.items(), key=lambda x: x[1], reverse=True)[:5]

In [ ]:
for e in GD.edges():
    GD[e[0]][e[1]]["betweeness centrality"] = betcent_e[e]

In [ ]:
# the bigger is betweeness centrality - the darker are edges
c = nv.CircosPlot(GD, edge_color="betweeness centrality")
c.draw()

In [ ]:
elbc = sorted(betcent_e.items(), key=lambda x: x[1], reverse=True)[:1][0][0]
palette = ["#bd3c14", "#009688"]
edge_colors = [palette[0] if e != elbc else palette[1] for e in GD.edges()]

In [ ]:
nx.draw(GD, edge_color=edge_colors, with_labels=True, node_color="#999999")


الحافة الخضراء هي التي تتمتع بأكبر مركزية بينية. الألوان مميزة تماما، وآمل :-)



### مركزية المتجهات الذاتية



حسنًا، لقد حاولت تجنب ذلك ولكن إليك صيغة حقيقية بدون كلمات:
$$ Ax = \lambda x $$A هي مصفوفة مجاورة للرسم البياني، (واحدة من التمثيلات الأكثر استخدامًا للرسم البياني (إن لم تكن الوحيدة) - مصفوفة NxN حيث تتوافق القيم الموجودة في تقاطعات الأعمدة والصفوف مع الاتصال بين العقد المعنية. تشير القيم غير الصفرية إلى الاتصال بين العقد (في حالة الرسم البياني غير الموزون، يكون 1 هو الخيار الوحيد) ويعني الصفر أن العقد غير متصلة بشكل مباشر.
$ \lambda $ هي القيمة الذاتية لـ A وأكبر قيمة ذاتية مرتبطة بالمتجه الذاتي لمصفوفة المجاورة A 
تعتمد أهمية العقدة على أهمية جيرانها. 


In [ ]:
eigcent = nx.eigenvector_centrality_numpy(GD)
sorted(eigcent.items(), key=lambda x: x[1], reverse=True)[:5]

In [ ]:
for n in GD.nodes():
    GD.node[n]["eigenvector centrality"] = eigcent[n]

In [ ]:
# you can use this function for colouring nodes according to their values of given attribute
def attribute_color(G, attribute):
    attrs = [G.node[n][attribute] for n in G.nodes()]
    uattrs = sorted(list(set(attrs)))
    palette = sns.color_palette("Blues", len(uattrs)).as_hex()
    colmap = dict(zip(uattrs, palette))
    node_colors = [colmap[at] for at in attrs]
    nx.draw(G, node_color=node_colors, with_labels=True)


دعونا نرى جميع أنواع المركزية الأربعة في صورة واحدة. تتوافق الظلال الداكنة من اللون الأزرق مع قيم مترية أكبر


In [ ]:
fig = plt.figure(figsize=(12, 12))

plt.subplot(221)
attribute_color(GD, "degree centrality")
plt.subplot(222)
attribute_color(GD, "closeness centrality")
plt.subplot(223)
attribute_color(GD, "betweeness centrality")
plt.subplot(224)
attribute_color(GD, "eigenvector centrality")


## توقع الارتباط



إذا كانت لدينا معلومات حول الشبكة الاجتماعية في هذه اللحظة المحددة، فقد نرغب في محاولة التنبؤ بالتفاعل الجديد الذي من المرجح أن يحدث في المستقبل.



### معامل الجاكار
للعثور على زوج من العقد "المتشابهة" مع بعضها البعض، يمكننا استخدام معامل جاكارد 
إنه يقيس نسبة الجيران الذين يتشاركهم زوج من العقد.



$$Jaccard _{uv}  = \frac { |N_u \cap N_v |} {|N_u\ \cup \ N_v|}$$
حيث $N_u$ - مجموعة جيران العقدة $u$


In [ ]:
jc = nx.jaccard_coefficient(GD)
jcd = {}
for u, v, p in jc:
    jcd[(u, v)] = p

In [ ]:
len(jcd)


ما عدد أزواج العقد التي لديها احتمال أكبر من 0.5 للاتصال في المستقبل القريب؟


In [ ]:
len([(u, v) for u, v in jcd.keys() if jcd[(u, v)] > 0.5])

In [ ]:
# let's see those pairs
[(u, v) for u, v in jcd.keys() if jcd[(u, v)] > 0.5]

In [ ]:
# let's vizualize nodes that are most likely to interact the same way as with nodes before
ejc = sorted(jcd.items(), key=lambda x: x[1], reverse=True)[:1][0][0]
palette = sns.color_palette("coolwarm", 2).as_hex()
node_colors = [palette[0] if n not in ejc else palette[1] for n in GD.nodes()]

In [ ]:
nx.draw(GD, node_color=node_colors, with_labels=True)


### المرفقات التفضيلية
وفي النهج الآخر لربط عقد التنبؤ بدرجة عالية، ستكون تلك التي من المرجح أن تحصل على اتصالات مستقبلية. 



$$
PA _{uv}  = { |N_u \cap N_v |} \ or \ {deg_u * deg_v}
$$


In [ ]:
pa = nx.preferential_attachment(GD)
pad = {}
for u, v, p in pa:
    pad[(u, v)] = p

In [ ]:
sorted(pad.items(), key=lambda x: x[1], reverse=True)[:5]


يمكنك توسيع نطاقه رغم ذلك :-)


In [ ]:
epa = sorted(pad.items(), key=lambda x: x[1], reverse=True)[:1][0][0]
palette = sns.color_palette("coolwarm", 2).as_hex()
node_colors = [palette[0] if n not in epa else palette[1] for n in GD.nodes()]

In [ ]:
nx.draw(GD, node_color=node_colors, with_labels=True)


### تخصيص الموارد
$$RA _{uv}  =  \sum_{w\in N_u \cap N_v} { \frac { 1} {deg_w } }$$فيما يلي الإلهام لاقتراح مقياس التشابه هذا: نحن نعتبر عقدتين $u$ و$v$، حيث يمكن لـ $u$ تخصيص الموارد لـ $v$ (بالإضافة إلى العكس)، من خلال جيرانهم المشتركين (أجهزة الإرسال). نحن نفترض أن كل عقدة لديها مورد واحد فقط تقوم بتخصيصه لجيرانها بالتساوي. وبالتالي يتم استخدام التعبير أعلاه لحساب مقدار الموارد التي يمكن للعقدة الحصول عليها من العقدة الأخرى. 
هناك طريقة مشابهة جدًا مقترحة لقياس احتمالية التفاعل - *Adamic/Adar (AA)*. الفرق هو أن الأخير لديه لوغاريتم في المقام. لذا فإن RA يعاقب الجيران من الدرجة العالية أكثر.
[هنا](https://networkx.github.io/documentation/stable/reference/algorithms/link_prediction.html) يمكنك العثور على جميع فهارس التشابه المطبقة في NetworkX.


In [ ]:
ra = nx.resource_allocation_index(GD)
rad = {}
for u, v, p in ra:
    rad[(u, v)] = p

In [ ]:
sorted(rad.items(), key=lambda x: x[1], reverse=True)[:5]

In [ ]:
era = sorted(rad.items(), key=lambda x: x[1], reverse=True)[:1][0][0]
palette = sns.color_palette("coolwarm", 2).as_hex()
node_colors = [palette[0] if n not in era else palette[1] for n in GD.nodes()]

In [ ]:
nx.draw(GD, node_color=node_colors, with_labels=True, node_size=200)


## تحليل الهيكل
###المجموعات
الزمر في تحليل الشبكة - الرسوم البيانية المتصلة بالكامل (أو مجموعة فرعية من العقد). التجمعات القصوى هي تلك المجموعات التي تتوقف عن كونها مجموعات مع إضافة أي حافة واحدة.


In [ ]:
len(list(nx.find_cliques(GD)))

In [ ]:
list(nx.find_cliques(GD))[:5]


نعم، الحافة تقع أيضًا ضمن تعريف الزمرة.
ولكي تعلم أن هذه الخوارزمية ليست مناسبة للرسم البياني الموجه (كما تنص [الوثائق](https://networkx.github.io/documentation/networkx-1.9/reference/generated/networkx.algorithms.clique.find_cliques.html))


In [ ]:
from networkx.algorithms.community import k_clique_communities


*k_clique_communities* تعرض قائمة بالمجموعات، تم إنتاج كل منها من خلال الجمع بين جميع مجموعات الحجم k التي تشارك العقد $k-1$. يمثل هذا مفهوم *طريقة الترشيح الجماعي* التي تسمح باكتشاف البنية المتداخلة للشبكة من خلال اكتشاف الرسوم البيانية الفرعية المتصلة بالكامل للعقد $k$. مجموعتان $k$ - تعتبر المجموعات متجاورة إذا كانت تشترك في $k$ مع الجيران.


In [ ]:
list(k_clique_communities(GD, 3, cliques=None))


### كشف المجتمعيبدو أن المجموعات المختلفة من العقد تكون مرتبطة بشكل أكثر كثافة داخليًا وليس خارجيًا، مما يجعلها تسمى "المجتمعات". في العقد الماضي، جذبت مشكلة تقسيم الرسم البياني إلى عدد من المجموعات الكثير من الاهتمام من جانب علماء الفيزياء والإحصائيين. يتضمن هذا المجال من الدراسة، والذي يُطلق عليه عادةً "اكتشاف المجتمع"، عددًا متزايدًا من الأوراق البحثية التي تقترح خوارزميات وطرق جديدة وتعديلات على الخوارزميات الموجودة. إن أهمية اكتشاف المجتمع واضحة تمامًا لأنها تسمح بتحليل بنية الشبكات، وجعل التصور أكثر وضوحًا، والتوصل إلى استنتاجات أو تنبؤات معينة بناءً على التحليل. تجدر الإشارة إلى أنه قبل الكشف، يكون هناك عدد من المجتمعات غير مؤكد، مما يشكل فرقًا بين اكتشاف المجتمع والتجمع، حيث يتم إعطاء كمية من المجموعات عن قصد. علاوة على ذلك، يمكن أن تختلف المجتمعات من حيث الحجم والكثافة، ويمكن أن تتضمن بنية هرمية. 



لذلك، تحتوي حزمة NetworkX للكشف عن المجتمع على خوارزمية تعتمد على [هذه الورقة](https://arxiv.org/pdf/cond-mat/0112110.pdf).


In [ ]:
from networkx.algorithms.community import girvan_newman


فكرة خوارزمية جيرفان نيومان هي أنه في كل خطوة تتم إزالة الحواف ذات أعلى مسافة بينية وبالتالي فصل الرسم البياني إلى أجزاء متصلة بكثافة.


In [ ]:
gn = girvan_newman(GD)
# returns an iterator over tuples of communities.


لذا فإن كل صف هو جزء من الرسم البياني عند المستوى الحالي للخوارزمية. في التكرار الأخير، جميع العقد نفسها هي مجتمعات:


In [ ]:
len(list(gn)[-1])
# as you might remember there are 62 nodes in our network


دعونا نرى حالة التقسيم في التكرار الثاني:


In [ ]:
gn = girvan_newman(GD)
next(gn)
second_iteration = tuple(sorted(c) for c in next(gn))

In [ ]:
print(second_iteration)

In [ ]:
from seaborn import color_palette


باستخدام هذه الوظيفة، يمكنك إنشاء تصور ملون للمجتمعات:


In [ ]:
def colour_communities(G, partition):
    commap = {}
    for n in G.nodes():
        for i, c in enumerate(partition):
            if n in c:
                commap[n] = i
                G.node[n]["community"] = i
    cs = [G.node[n]["community"] for n in G.nodes()]
    ucs = list(set(cs))
    palette = color_palette("coolwarm", len(ucs)).as_hex()
    colmap = dict(zip(ucs, palette))
    node_colors = [colmap[c] for c in cs]

    return node_colors, colmap, palette

In [ ]:
# store partitions at first four iterations
gn = girvan_newman(GD)
first_iteration = tuple(sorted(c) for c in next(gn))
second_iteration = tuple(sorted(c) for c in next(gn))
third_iteration = tuple(sorted(c) for c in next(gn))
fourth_iteration = tuple(sorted(c) for c in next(gn))

In [ ]:
[len(part) for part in list(first_iteration)]


التكرار الأول - مجتمعان يتكونان من 41 و21 عقدة.


In [ ]:
display([len(part) for part in list(second_iteration)])
display([len(part) for part in list(third_iteration)])
display([len(part) for part in list(fourth_iteration)])


الآن الصور!


In [ ]:
fig = plt.figure(figsize=(20, 15))


plt.subplot(221)
node_colors, color_map, palette = colour_communities(GD, first_iteration)
nx.draw(GD, node_color=node_colors, with_labels=True)
plt.subplot(222)
node_colors, color_map, palette = colour_communities(GD, second_iteration)
nx.draw(GD, node_color=node_colors, with_labels=True)
plt.subplot(223)
node_colors, color_map, palette = colour_communities(GD, third_iteration)
nx.draw(GD, node_color=node_colors, with_labels=True)
plt.subplot(224)
node_colors, color_map, palette = colour_communities(GD, fourth_iteration)
nx.draw(GD, node_color=node_colors, with_labels=True)


### رسم بياني فرعيعندما يكون لديك رسم بياني كبير ولكنك ترغب في تصور أو تحليل جزء صغير منه فقط، فقد يكون من المفيد استخراج العقد محل الاهتمام والحواف المقابلة. هذا ما يدور حوله الرسم البياني الفرعي.
على سبيل المثال، لنقم بإنشاء رسم بياني فرعي استنادًا إلى العقدة ذات الدرجة الأكبر وجيرانها


In [ ]:
sorted(dict(GD.degree()).items(), key=lambda x: x[1], reverse=True)[:1]

In [ ]:
nbrs = list(GD.neighbors("15"))
nbrs

In [ ]:
# don't forget to include your node of interest
nbrs.append("15")

In [ ]:
GD_sub15 = GD.subgraph(nbrs)
print(nx.info(GD_sub15))

In [ ]:
nx.draw(GD_sub15, with_labels=True, alpha=0.7, node_size=500)


## الرسوم البيانية الثنائية



الرسوم البيانية الثنائية - الرسوم البيانية التي:
 - مقسمة إلى مجموعتين (كل عقدة تنتمي إلى إحدى المجموعتين)
 - يمكن توصيل العقدة فقط بعقدة من مجموعة أخرى
 
 
 كمثال على الرسم البياني ثنائي الأجزاء، يمكنك تخيل رسم بياني يتكون من مجموعتين: العملاء والمنتجات التي يشترونها، ومستخدمو موقع الويب والأفلام التي يتركون تعليقاتهم عليها.



في NetworkX، يمكن تنفيذ المفهوم الثنائي عن طريق الكلمة الرئيسية *bipartite* عند تحديد البيانات الوصفية للعقد (أو عن طريق أي اسم سمة تريده).



الآن أخذت [الرسم البياني](https://www.dropbox.com/s/mnxzbndbxq1dif5/out.brunson_club-membership_club-membership?dl=0) من [نفس المصدر](http://konect.uni-koblenz.de/networks/). إنه أيضًا مثال حقيقي ولكنه لا يزال لعبة (بطريقة ما). تتكون الشبكة من 40 عقدة - 25 مسؤولًا تنفيذيًا للشركات و15 منظمة اجتماعية هم أعضاء فيها. وكل حافة (بمبلغ 95) تشير إلى أن الشخص الحالي لديه حالة عضوية في النادي المقابل.


In [ ]:
edges = []
with open("datasets/out.brunson_club-membership_club-membership", "r") as f:
    filereader = csv.reader(f, delimiter=" ", quotechar='"')
    next(filereader)  # skips header row
    next(filereader)
    for row in filereader:
        edges.append(row[:2])

In [ ]:
# in original format each set of nodes begins from 1.. so for the sake of distinguishability this is what I came to
for edge in edges:
    edge[0] = "m" + edge[0]
    edge[1] = "c" + edge[1]

In [ ]:
edges[:5]

In [ ]:
CM = nx.Graph()

In [ ]:
clubs = ["c" + str(i + 1) for i in range(15)]
members = ["m" + str(i + 1) for i in range(25)]

In [ ]:
CM.add_nodes_from(clubs, bipartite="clubs")
CM.add_nodes_from(members, bipartite="members")
CM.add_edges_from(edges)

In [ ]:
top = nx.bipartite.sets(CM)[0]
pos = nx.bipartite_layout(CM, top, scale=7)

In [ ]:
print(nx.info(CM))


يمكن تصور الرسوم البيانية الثنائية على النحو التالي:


In [ ]:
fig = plt.figure(figsize=(7, 7))
nx.draw(
    CM,
    pos,
    node_color="#9699bc",
    alpha=0.7,
    with_labels=True,
    aspect_ratio=0.1,
    node_size=450,
)


حان الوقت الآن لعرض تطبيق (نوع) الرسم البياني ثنائي القطب - التوصيات. بالإضافة إلى الرسم البياني، ستكون مهمتنا لعبة نسبيًا. دعنا نحاول أن نوصي الشخص بمنظمة للانضمام إليها بناءً على أشخاص مشابهين. (يتم قياس التشابه بين الأشخاص على أساس المنظمة التي ينتمون إليها.


In [ ]:
# function that produces the set of clubs both persons are going to
def shared_partition(G, mbr1, mbr2):
    nbrs1 = G.neighbors(mbr1)
    nbrs2 = G.neighbors(mbr2)
    overlap = set(nbrs1).intersection(nbrs2)
    return overlap


def member_similarity(G, mbr1, mbr2):
    shared_nodes = shared_partition(G, mbr1, mbr2)
    return len(shared_nodes) / len(clubs)

In [ ]:
shared_partition(CM, "m3", "m5")

In [ ]:
member_similarity(CM, "m23", "m13")

In [ ]:
def most_similar_members(G, member, thr=0.14):  # you can specify the threshold
    mbrs = members.copy()
    mbrs.remove(member)
    similarities = defaultdict(list)
    for m in mbrs:
        similarity = member_similarity(G, member, m)
        similarities[similarity].append(m)

    max_similarity = max(similarities.keys())
    if max_similarity >= thr:
        return [similarities[max_similarity]]
    else:
        return [[]]

In [ ]:
most_similar_members(CM, "m15")

In [ ]:
def recommend_club(G, member, thr=0.14):
    smlr = most_similar_members(G, member, thr)
    tmp = [
        list(G.neighbors(n)) for n in smlr[0]
    ]  # we make a list of neighbours lists for every node in list of similar members
    smlr_clubs = set([i for sub in tmp for i in sub])  # ..and then flatten it
    mbr_clubs = set(G.neighbors(member))
    return list(smlr_clubs.difference(mbr_clubs))


print(recommend_club(CM, "m15"))

In [ ]:
# store recommendations for every member
recommendations = {}
for mbr in members:
    recommendations[mbr] = [recommend_club(CM, mbr)]

In [ ]:
recommendations

كما قلت، إنه مثال لعبة جدًا، ولهذا السبب فإن الحد الافتراضي (0.14) صغير جدًا. لكن أعتقد أنك قد استوعبت فكرة عما يمكن القيام به. هناك أيضًا العديد من الأوراق البحثية المخصصة لربط التنبؤ في الرسوم البيانية الثنائية ولكن هذا بالفعل خارج نطاق هذا البرنامج التعليمي



هذا كل شيء :-) أتمنى أن تكون قد تعلمت شيئًا جديدًا من هذا. 
NetworkX عبارة عن مجموعة من الأدوات المتنوعة لدراسة بنية الشبكة وديناميكياتها والتي يمكنك استخدامها لتلبية احتياجاتك للعثور على رؤية مفيدة لبياناتك. يعد تحليل الشبكة بدوره أسلوبًا مفيدًا لتحليل الأنظمة ويمكن تنفيذه في علم الأحياء والعلوم الاجتماعية والخدمات اللوجستية. علاوة على ذلك، فإن اكتشاف المجتمع موجود منذ أقل من 20 عامًا ويتم اقتراح خوارزميات/طرق جديدة في الوقت الحاضر. 



هنا يمكنك أن تجد شيئا مفيدا:
 - [دورة في DataCamp](https://www.datacamp.com/courses/network-analysis-in-python-part-1)
 - [فيديو العرض التقديمي في PyData Carolinas 2016](https://www.youtube.com/watch?v=7fsreJMy_pI)
 - [وثائق NetworkX](https://networkx.github.io/documentation/stable/reference/index.html)
 - [أول ورقة بحثية مخصصة لـ *اكتشاف المجتمع* على ما أعتقد](https://arxiv.org/pdf/cond-mat/0112110.pdf)
 - [موضوع آخر حول اكتشاف المجتمع ولكن دراسة فقط](https://arxiv.org/pdf/0906.0612.pdf)
 - [مجموعة الشبكات، مرة أخرى](http://konect.uni-koblenz.de/networks/)